In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder (ResNet18 adapted for CIFAR10)
        resnet = models.resnet18(pretrained=False)
        resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)  # Adapt first layer
        resnet.maxpool = nn.Identity()  # Remove initial maxpool
        self.encoder = nn.Sequential(*list(resnet.children())[:-1])
        
        # Projection head
        self.projection = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 128))
        
        # Decoder for 32x32 output
        self.decoder = nn.Sequential(
            nn.Linear(512, 512 * 4 * 4),
            nn.Unflatten(1, (512, 4, 4)),
            nn.ConvTranspose2d(512, 256, 4, 2, 1),  # 4x4 -> 8x8
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1),  # 8x8 -> 16x16
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),   # 16x16 -> 32x32
            nn.ReLU(),
            nn.Conv2d(64, 3, 3, 1, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        features = self.encoder(x).flatten(1)
        projected = F.normalize(self.projection(features), p=2, dim=1)
        reconstruction = self.decoder(features)
        return reconstruction, projected

In [3]:
# class Autoencoder(nn.Module):
#     def __init__(self):
#         super().__init__()
#         # Enhanced ResNet18 encoder for CIFAR10
#         resnet = models.resnet18(pretrained=False)
#         resnet.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
#         resnet.maxpool = nn.Identity()
#         self.encoder = nn.Sequential(*list(resnet.children())[:-1])
        
#         # Stronger projection head
#         self.projection = nn.Sequential(
#             nn.Linear(512, 1024),
#             nn.BatchNorm1d(1024),
#             nn.ReLU(),
#             nn.Linear(1024, 256),
#             nn.BatchNorm1d(256),
#             nn.ReLU(),
#             nn.Linear(256, 128)
#         )

#         # More powerful decoder
#         self.decoder = nn.Sequential(
#             nn.Linear(512, 512 * 4 * 4),
#             nn.Unflatten(1, (512, 4, 4)),
#             nn.ConvTranspose2d(512, 256, 4, 2, 1),  # 8x8
#             nn.BatchNorm2d(256),
#             nn.ReLU(),
#             nn.ConvTranspose2d(256, 128, 4, 2, 1),  # 16x16
#             nn.BatchNorm2d(128),
#             nn.ReLU(),
#             nn.ConvTranspose2d(128, 64, 4, 2, 1),   # 32x32
#             nn.BatchNorm2d(64),
#             nn.ReLU(),
#             nn.Conv2d(64, 3, 3, 1, 1),
#             nn.Sigmoid()
#         )
#     def forward(self, x):
#         features = self.encoder(x).flatten(1)
#         projected = F.normalize(self.projection(features), p=2, dim=1)
#         reconstruction = self.decoder(features)
#         return reconstruction, projected

In [4]:
class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        batch_size = features.shape[0]
        
        # Compute similarity matrix
        similarity_matrix = torch.matmul(features, features.T) / self.temperature
        
        # Create mask for positive pairs (excluding self)
        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)
        self_mask = torch.eye(batch_size, dtype=torch.float32).to(device)
        mask = mask - self_mask
        
        # Subtract max for numerical stability
        max_sim, _ = torch.max(similarity_matrix, dim=1, keepdim=True)
        logits = similarity_matrix - max_sim.detach()
        
        # Compute log probabilities
        exp_logits = torch.exp(logits)
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-10)
        
        # Compute mean log prob over positive pairs
        mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-10)
        mean_log_prob_pos = torch.nan_to_num(mean_log_prob_pos, nan=0.0)
        
        loss = -mean_log_prob_pos.mean()

        return loss

In [5]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

from torch.utils.data import Dataset
class MemoryDataset(Dataset):
    def __init__(self, dataset):
        self.data = [dataset[i] for i in range(len(dataset))]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

# Define data transformations
# train_transform = transforms.Compose([
#     transforms.RandomResizedCrop(32, scale=(0.8, 1.0)),  # Random crop
#     transforms.RandomHorizontalFlip(),
#     transforms.ToTensor(),
#     transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))  # CIFAR10 stats
# ])
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomAffine(0, shear=10),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

# Load CIFAR10 dataset
train_val_dataset = datasets.CIFAR10(root='./data', 
                                    train=True, 
                                    download=True,
                                    transform=train_transform)

# Split into train and validation (90% train, 10% val)
train_size = int(0.9 * len(train_val_dataset))
val_size = len(train_val_dataset) - train_size
train_dataset, val_dataset = random_split(train_val_dataset, [train_size, val_size])

# Replace validation set transform (no augmentation)
val_dataset.dataset.transform = val_transform


train_dataset = MemoryDataset(train_dataset) # loads dataset to memory!
val_dataset = MemoryDataset(val_dataset)
# Create dataloaders
batch_size = 128
train_loader = DataLoader(train_dataset, 
                         batch_size=batch_size,
                         shuffle=True,
                         num_workers=0,
                         pin_memory=True)

val_loader = DataLoader(val_dataset,
                       batch_size=batch_size,
                       shuffle=False,
                       num_workers=0,
                       pin_memory=True)

# Optional: Test dataset (not used in training)
test_dataset = datasets.CIFAR10(root='./data',
                               train=False,
                               download=True,
                               transform=val_transform)

test_dataset = MemoryDataset(test_dataset)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Print dataset sizes
print(f'Train samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')
print(f'Test samples: {len(test_dataset)}')

Train samples: 45000
Validation samples: 5000
Test samples: 10000


In [ ]:
# Initialize model and loss functions
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Autoencoder().to(device)
supcon_criterion = SupConLoss(temperature=0.07)
recon_criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0022, weight_decay=1e-4)
# Training loop
num_epochs = 15
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)
contrastive_weight = 0.5  # Adjust based on your needs

c:\Users\gal19\anaconda3\envs\cs236781-hw\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\gal19\anaconda3\envs\cs236781-hw\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
c:\Users\gal19\anaconda3\envs\cs236781-hw\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [ ]:
# import torch_lr_finder

# # Run this before training
# lr_finder = torch_lr_finder.LRFinder(model, optimizer, criterion)
# lr_finder.range_test(train_loader, end_lr=1e-2, num_iter=100)
# lr_finder.plot()

In [18]:

print("start epochs")
best_val_loss = float('inf')
for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0.0
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        
        # Forward pass
        reconstructions, projections = model(inputs)
        recon_loss = recon_criterion(reconstructions, inputs)
        supcon_loss = supcon_criterion(projections, labels)
        total_loss = (1 - contrastive_weight) * recon_loss + contrastive_weight * supcon_loss
        
        # Backward pass and optimize
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        train_loss += total_loss.item()
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            reconstructions, projections = model(inputs)
            recon_loss = recon_criterion(reconstructions, inputs)
            supcon_loss = supcon_criterion(projections, labels)
            total_loss = recon_loss + contrastive_weight * supcon_loss
            
            val_loss += total_loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    scheduler.step(avg_val_loss)
    # Save model when validation loss improves
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model_paper.pth')
        print("saved model")
    
    print(f'Epoch [{epoch+1}/{num_epochs}]')
    print(f'Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}')

start epochs
saved model
Epoch [1/15]
Train Loss: 2.7638, Val Loss: 3.1744
saved model
Epoch [2/15]
Train Loss: 2.6763, Val Loss: 3.0943
saved model
Epoch [3/15]
Train Loss: 2.6239, Val Loss: 3.0647
saved model
Epoch [4/15]
Train Loss: 2.5554, Val Loss: 3.0010
saved model
Epoch [5/15]
Train Loss: 2.4647, Val Loss: 2.9498
saved model
Epoch [6/15]
Train Loss: 2.3727, Val Loss: 2.8900
saved model
Epoch [7/15]
Train Loss: 2.2941, Val Loss: 2.7841
Epoch [8/15]
Train Loss: 2.2159, Val Loss: 2.7881
Epoch [9/15]
Train Loss: 2.1476, Val Loss: 2.7843
saved model
Epoch [10/15]
Train Loss: 2.0862, Val Loss: 2.7792
Epoch [11/15]
Train Loss: 2.0364, Val Loss: 2.7811
Epoch [12/15]
Train Loss: 1.9943, Val Loss: 2.8049
Epoch [13/15]
Train Loss: 1.9619, Val Loss: 2.7869
Epoch [14/15]
Train Loss: 1.9389, Val Loss: 2.8493
Epoch [15/15]
Train Loss: 1.8847, Val Loss: 2.8483


In [9]:
# model.load_state_dict(torch.load('best_model_paper.pth'))
model.load_state_dict(torch.load('best_model_paper_final.pth'))
model.eval()

Autoencoder(
  (encoder): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Identity()
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(

In [10]:
# Visualization
import importlib
import utils
importlib.reload(utils)
from utils import plot_tsne
plot_tsne(model.encoder, test_loader, device)

In [47]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
myencoder = model
# Assuming `myencoder` is your pretrained encoder, set to evaluation mode
myencoder.eval()

# Define your dataset and DataLoaders (adjust based on your dataset)
# Example using MNIST:
from torchvision import datasets, transforms
# transform = transforms.Compose([transforms.ToTensor()])
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])
train_val_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
batch_size = 256

train_size = int(0.8 * len(train_val_dataset))
val_size = len(train_val_dataset) - train_size
train_dataset, val_dataset = random_split(train_val_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Function to extract latent representations
def extract_latent(loader, model, device):
    model.to(device)
    latent_vectors = []
    labels = []
    with torch.no_grad():
        for data, target in loader:
            data = data.to(device)
            features = model.encoder(data).flatten(1)
            latent = model.projection(features).cpu().numpy()
            latent_vectors.append(latent)
            labels.extend(target.numpy())
    latent_vectors = np.vstack(latent_vectors)
    labels = np.array(labels)
    return latent_vectors, labels

# Extract latent representations
X_train, y_train = extract_latent(train_loader, myencoder, device)
X_test, y_test = extract_latent(test_loader, myencoder, device)
X_val, y_val = extract_latent(val_loader, myencoder, device)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create DataLoaders for latent space
train_dataset_latent = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset_latent = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset_latent = TensorDataset(X_test_tensor, y_test_tensor)
val_loader_latent = DataLoader(val_dataset_latent, batch_size=batch_size, shuffle=False)
train_loader_latent = DataLoader(train_dataset_latent, batch_size=batch_size, shuffle=True)
test_loader_latent = DataLoader(test_dataset_latent, batch_size=batch_size, shuffle=False)


In [ ]:
class Classifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(Classifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, num_classes)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:



# Initialize classifier (replace input_dim with your latent dimension)
input_dim = 128  # Adjust this based on myencoder's output size
num_classes = 10
# classifier = Classifier(input_dim, num_classes).to(device)
classifier = Classifier(input_dim, num_classes).to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(classifier.parameters(), lr=0.001)

best_val_acc = 0.0
num_epochs = 15

for epoch in range(num_epochs):
    # Training phase
    classifier.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for inputs, labels in train_loader_latent:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = classifier(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        # scheduler.step()
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total_train += labels.size(0)
        correct_train += predicted.eq(labels).sum().item()
    
    # Validation phase
    classifier.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader_latent:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = classifier(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            total_val += labels.size(0)
            correct_val += predicted.eq(labels).sum().item()
    
    # Calculate metrics
    train_acc = 100. * correct_train / total_train
    val_acc = 100. * correct_val / total_val
    avg_train_loss = train_loss / len(train_loader_latent)
    avg_val_loss = val_loss / len(val_loader_latent)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(classifier.state_dict(), 'best_classifier_new.pth')
    
    print(f'Epoch {epoch+1}/{num_epochs}')
    print(f'Train Loss: {avg_train_loss:.4f} | Acc: {train_acc:.2f}%')
    print(f'Val Loss: {avg_val_loss:.4f} | Acc: {val_acc:.2f}%')
    print('-' * 50)
    


# Evaluation
classifier.eval()



# Load best model
classifier.load_state_dict(torch.load('best_classifier_new.pth'))
classifier.eval()

test_loss = 0.0
correct_test = 0
total_test = 0

with torch.no_grad():
    for inputs, labels in test_loader_latent:
        inputs, labels = inputs.to(device), labels.to(device)
        
        outputs = classifier(inputs)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item()
        _, predicted = outputs.max(1)
        total_test += labels.size(0)
        correct_test += predicted.eq(labels).sum().item()

avg_test_loss = test_loss / len(test_loader_latent)
test_acc = 100. * correct_test / total_test

print(f'Test Results:')
print(f'Loss: {avg_test_loss:.4f} | Accuracy: {test_acc:.2f}%')

Epoch 1/15
Train Loss: 1.3400 | Acc: 84.08%
Val Loss: 0.5036 | Acc: 87.74%
--------------------------------------------------
Epoch 2/15
Train Loss: 0.5259 | Acc: 86.45%
Val Loss: 0.4728 | Acc: 86.89%
--------------------------------------------------
Epoch 3/15
Train Loss: 0.4458 | Acc: 86.71%
Val Loss: 0.4099 | Acc: 87.25%
--------------------------------------------------
Epoch 4/15
Train Loss: 0.4510 | Acc: 87.07%
Val Loss: 0.3580 | Acc: 88.91%
--------------------------------------------------
Epoch 5/15
Train Loss: 0.4096 | Acc: 86.99%
Val Loss: 0.4127 | Acc: 86.45%
--------------------------------------------------


KeyboardInterrupt: 